 Students have to propose a new model that can help the buyer to predict the price trends without official information from the airlines. 

In [1]:
#import libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

In [2]:
#load the dataset

data = pd.read_excel(r'C:\Users\Shreya\OneDrive\Desktop\Projects\Airfare price prediction\Data_Train.xlsx')

In [3]:
# check the data
data.head()

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price
0,IndiGo,24/03/2019,Banglore,New Delhi,BLR → DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897
1,Air India,1/05/2019,Kolkata,Banglore,CCU → IXR → BBI → BLR,05:50,13:15,7h 25m,2 stops,No info,7662
2,Jet Airways,9/06/2019,Delhi,Cochin,DEL → LKO → BOM → COK,09:25,04:25 10 Jun,19h,2 stops,No info,13882
3,IndiGo,12/05/2019,Kolkata,Banglore,CCU → NAG → BLR,18:05,23:30,5h 25m,1 stop,No info,6218
4,IndiGo,01/03/2019,Banglore,New Delhi,BLR → NAG → DEL,16:50,21:35,4h 45m,1 stop,No info,13302


In [4]:
data.shape

(10683, 11)

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Airline          10683 non-null  object
 1   Date_of_Journey  10683 non-null  object
 2   Source           10683 non-null  object
 3   Destination      10683 non-null  object
 4   Route            10682 non-null  object
 5   Dep_Time         10683 non-null  object
 6   Arrival_Time     10683 non-null  object
 7   Duration         10683 non-null  object
 8   Total_Stops      10682 non-null  object
 9   Additional_Info  10683 non-null  object
 10  Price            10683 non-null  int64 
dtypes: int64(1), object(10)
memory usage: 918.2+ KB


In [6]:
data.isnull().sum()

Airline            0
Date_of_Journey    0
Source             0
Destination        0
Route              1
Dep_Time           0
Arrival_Time       0
Duration           0
Total_Stops        1
Additional_Info    0
Price              0
dtype: int64

In [7]:
data["Journey_Day"] = pd.to_datetime(data["Date_of_Journey"], format="%d/%m/%Y").dt.day

In [8]:
data["Journey_Month"] = pd.to_datetime(data["Date_of_Journey"], format="%d/%m/%Y").dt.month

In [9]:
data.drop("Date_of_Journey", axis=1, inplace=True)

In [10]:
data["Dep_Hour"] = pd.to_datetime(data["Dep_Time"]).dt.hour

data["Dep_Min"] = pd.to_datetime(data["Dep_Time"]).dt.minute

In [11]:
data.drop("Dep_Time", axis=1, inplace=True)

In [12]:
data["Arrival_Hour"] = pd.to_datetime(data["Arrival_Time"].str.split(' ').str[0]).dt.hour

data["Arrival_Min"] = pd.to_datetime(data["Arrival_Time"].str.split(' ').str[0]).dt.minute

In [13]:
data.drop("Arrival_Time", axis=1, inplace=True)

In [14]:
Duration_Hours = []
Duration_Mins = []

In [15]:
for x in data["Duration"]:
    h = 0
    m = 0

    parts = x.split()

    for p in parts:
        if 'h' in p:
            h = int(p.replace('h', ''))

        if 'm' in p:
            m = int(p.replace('m', ''))

    Duration_Hours.append(h)
    Duration_Mins.append(m)

In [16]:
data["Duration_Hours"] = Duration_Hours

data["Duration_Mins"] = Duration_Mins

In [17]:
data.drop("Duration", axis=1, inplace=True)

In [18]:
data["Total_Stops"] = data["Total_Stops"].map({
    'non-stop': 0,
    '1 stop': 1,
    '2 stops': 2,
    '3 stops': 3,
    '4 stops': 4
})

In [19]:
X = data.drop("Price", axis=1)

y = data["Price"]

In [20]:
##Identify Categorical and Numerical Columns

categorical_columns = X.select_dtypes(include='object').columns

numerical_columns = X.select_dtypes(exclude='object').columns

In [21]:
print(categorical_columns)
print(numerical_columns)

Index(['Airline', 'Source', 'Destination', 'Route', 'Additional_Info'], dtype='object')
Index(['Total_Stops', 'Journey_Day', 'Journey_Month', 'Dep_Hour', 'Dep_Min',
       'Arrival_Hour', 'Arrival_Min', 'Duration_Hours', 'Duration_Mins'],
      dtype='object')


In [22]:
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_columns),
    ('num', SimpleImputer(strategy='median'), numerical_columns)
])

In [23]:
##Create Linear Regression Model

model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])


In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [25]:
model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [26]:
y_pred = model.predict(X_test)

In [27]:
print(y_pred[:10])

[ 9548.26544946  8671.59813982 14552.39745651  3900.85285509
 10033.99737335 11614.99130992 13254.15901149  9380.16957719
  3328.78740795 13574.25901731]


In [28]:
##Calculate RMSE

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("RMSE:", rmse)

RMSE: 2323.7023588493403


In [29]:
##Calculate R² Score

r2 = r2_score(y_test, y_pred)

print("R² Score:", r2)

R² Score: 0.7448685828395181


In [30]:
print("RMSE:", rmse)
print("R² Score:", r2)

RMSE: 2323.7023588493403
R² Score: 0.7448685828395181


In [31]:
##Predict New Flight Price

##Example:
sample = X_test.iloc[[0]]
prediction = model.predict(sample)
print("Predicted Price:", prediction)

Predicted Price: [9548.26544946]
